# AI Portal — Multi-Agent System

This notebook demonstrates the AI portal: a multi-agent system that routes user requests
to specialist Claude agents and optionally publishes results to external services.

## Architecture

```
User (Telegram / API)
  → Orchestrator
  → Registrar (classify & route)
  → Executor / Market Bot / CRM Connector
  → Google Docs / Tilda / KOMMO CRM (optional)
  → Reply
```

## Agents

| Agent | Model | Role |
|-------|-------|------|
| Registrar | claude-haiku-4-5 | Classify intent, choose agent |
| Executor | claude-sonnet-4-6 | Write documents, reports, code |
| Market Bot | claude-haiku-4-5 + Gemini | Ad copy, competitor analysis |
| CRM Connector | claude-haiku-4-5 | Create/update KOMMO leads |

## Integrations
- **Telegram** — user interface (bot)
- **Google Workspace** — Docs, Sheets, Drive
- **KOMMO CRM** — lead management
- **Tilda** — website pages


In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("."))))

from dotenv import load_dotenv

load_dotenv("../.env")
load_dotenv(".env")

print("Environment loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## 1. Storage Layer

All tasks and artifacts are stored in a local SQLite database.

In [ ]:
from portal.storage import create_task, init_db, list_tasks

# Initialise DB (creates tables if not exist)
init_db()
print("Database ready at portal/storage/portal.db")

## 2. Registrar Agent — Intent Classification

The Registrar uses the cheapest model (Haiku) to classify the user's request.

In [ ]:
from portal.agents.registrar import RegistrarAgent
from portal.storage import create_task

registrar = RegistrarAgent()

examples = [
    "Напиши рекламу для онлайн-школи англійської мови в Facebook",
    "Створи ліда в CRM: Олена Коваленко, тел +380501234567",
    "Зроби аналіз конкурентів у ніші онлайн-фітнесу",
    "Напиши звіт у Google Docs про результати маркетингу за Q1",
]

for text in examples:
    task_id = create_task(source="notebook", source_id="demo", input_text=text)
    result = registrar.classify(task_id, text)
    print(f"Input: {text[:60]}")
    print(f"  → Agent: {result['agent']} | Priority: {result['priority']}")
    print(f"  → Intent: {result['intent']}")
    print()

## 3. Orchestrator — End-to-End Flow

The Orchestrator chains all agents together automatically.

In [ ]:
from portal.orchestrator import Orchestrator

orch = Orchestrator()

# Example 1: Document generation
result = orch.handle(
    source="notebook",
    source_id="demo",
    text="Напиши короткий опис продукту для SaaS платформи управління проектами. Цільова аудиторія: малий бізнес",
)
print("=== EXECUTOR RESULT ===")
print(result[:800])

In [ ]:
# Example 2: Market analysis
result = orch.handle(
    source="notebook",
    source_id="demo",
    text="Зроби аналіз ринку онлайн-освіти в Україні: основні гравці, цінові сегменти, можливості",
)
print("=== MARKET BOT RESULT ===")
print(result[:800])

## 4. Task History

All tasks are saved in the local DB.

In [ ]:
from portal.storage import get_artifacts, list_tasks

tasks = list_tasks(source_id="demo", limit=10)
print(f"Total tasks: {len(tasks)}")
for t in tasks:
    arts = get_artifacts(t["id"])
    print(
        f"[{t['status']:6}] [{t.get('agent') or 'pending':15}] {t['input'][:60]} | artifacts: {len(arts)}"
    )

## 5. Running the Telegram Bot

After setting `TELEGRAM_BOT_TOKEN` in `.env`, run the bot:

In [ ]:
# This cell shows how to start the bot — don't run in notebook, use terminal instead
print("""
To start the Telegram bot:

  python -m portal.integrations.telegram_bot

Or in Python:
  from portal.integrations.telegram_bot import run_bot
  run_bot()

Bot commands:
  /start  — welcome
  /status — recent tasks
  /help   — capabilities

Then send any message like:
  "Напиши рекламу для фітнес клубу"
  "Створи ліда: Іван Іваненко +380671234567"
""")

## 6. Architecture: Adding a New Agent

To add a new specialist agent:

1. Create `portal/agents/my_agent.py` with a class that has `.run(task_id, intent, text, extracted) → str`
2. Register it in `portal/agents/__init__.py`
3. Add the routing label in `portal/agents/registrar.py` (SYSTEM_PROMPT)
4. Handle it in `portal/orchestrator.py` in the `_dispatch()` method

That's it — the rest of the system (storage, Telegram, logging) works automatically.